# Hard negatives across splits: the second sweep (Colab)

The 18-run grid measured its three effects on one ingredient split. The split-variance experiment (write-up §5.8) showed the split alone moves test acc@1 by about two points, which is the size of the hard-negatives effect (+3.0 for ingredient-matched over in-batch only). This sweep re-runs that one factor on all six splits: SapBERT × {ingredient, tfidf, none} × strength normalizer on × splits v1–v6 = 18 runs, paired within split and all on one machine. Config: `sweeps/negatives_by_split.yaml`.

**Before running:** Runtime → Change runtime type → **A100 GPU** (about a minute per run; a T4 takes about two). Add the `WANDB_API_KEY` secret (key icon, *Notebook access* on). The sweep is already registered; its id is in cell 5.

A grid sweep hands each combination out once. If a run crashes, the cell is not re-issued: rerun this cell and the analysis (`12_split_seeds.py negatives`) reports the cell as missing rather than failing.

## 1. Check the GPU
Expect a Tesla T4 (or better). If this prints nothing, the runtime type is still CPU.

In [ ]:
!nvidia-smi -L

## 2. Get the code
A plain clone of the public repo. `BRANCH` lets this run from a feature branch before it is merged; re-running the cell pulls the latest commit.

In [ ]:
import os, subprocess

REPO = "kvenanzi/rxnorm"
BRANCH = "main"
URL = f"https://github.com/{REPO}.git"
if not os.path.isdir("rxnorm"):
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, URL], check=True)
else:
    subprocess.run(["git", "-C", "rxnorm", "checkout", "-q", BRANCH], check=True)
    subprocess.run(["git", "-C", "rxnorm", "pull", "-q"], check=True)
!git -C rxnorm log --oneline -1

## 3. Make the package importable and install what Colab lacks
The clone directory goes on `sys.path`, so `import rxnorm_vandf` reads the code straight from the clone (a `git pull` in cell 2 is picked up immediately). An editable `pip install -e` would need a kernel restart to take effect in Colab.

The pip line adds only what Colab doesn't already ship, without upgrading what it does: upgrading Colab's numpy/pandas/torch inside a running kernel breaks its preinstalled stack.

In [ ]:
import sys
if os.path.abspath("rxnorm") not in sys.path:
    sys.path.insert(0, os.path.abspath("rxnorm"))
%pip install -q duckdb sentence-transformers datasets wandb accelerate
import torch, sentence_transformers, numpy, pandas, rxnorm_vandf
print("torch", torch.__version__, "cuda", torch.cuda.is_available(),
      "| sentence-transformers", sentence_transformers.__version__,
      "| numpy", numpy.__version__, "| pandas", pandas.__version__,
      "| rxnorm_vandf from", os.path.dirname(rxnorm_vandf.__file__))

## 4. Log in to Weights & Biases
The API key comes from Colab Secrets. `wandb.login()` reads the `WANDB_API_KEY` environment variable, so nothing is pasted or printed.

In [ ]:
import wandb
from google.colab import userdata

os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
wandb.login()

## 5. Start the agent
`run_one` builds a `TrainConfig` with the fixed choices (data from the artifact named in the sweep config, no model artifact per trial) and passes `sweep=True`, so `train()` takes this trial's parameters, including the split folder `dataset_subdir`, from `wandb.config`. `count=18` runs the whole grid; rerunning the cell after a disconnect picks up the untried cells.

In [ ]:
import gc
from rxnorm_vandf.train import TrainConfig, train

SWEEP_ID = "kettle-labs/rxnorm-vandf/uukeyzw7"

def run_one():
    train(TrainConfig(data_dir=None, output_dir="models", log_model=False,
                      tags=["sweep", "colab", "negatives-by-split"]), sweep=True)
    gc.collect(); torch.cuda.empty_cache()

wandb.agent(SWEEP_ID, function=run_one, count=18)

## 6. Afterwards
On the sweep page, group the runs table by `negatives` and by `dataset_subdir`. Back on the local machine, `uv run scripts/12_split_seeds.py negatives` writes the paired table (`outputs/split_seeds/negatives.md`): for each split, ingredient − none, tfidf − none, and ingredient − tfidf on test and on validation, with the mean, a 95% interval, and the sign count.